# Result-value edits under the reference-attention oracle, free generation (twelve carriers), Kaggle T4

Same Tesla T4, same Qwen3-4B revision, FP16 weights, greedy decoding, 384 plan / 48 answer tokens. Same nine cases, frozen construction, prompts and generic support. Oracle diagnostic, not a method. No adapter.

Branches (free generation, scored normally):
- **tk_tv**, **fk_fv**: ordinary generation (must reproduce refgen12)
- **tk_tv_refattn**: lockstep [fk_fv, tk_tv] with reference attention, intact values (must reproduce refgen12)
- **tk_tv_refattn_vzero**: same, target result-region values zeroed
- **tk_tv_refattn_vperm**: same, target result-region value rows permuted with the permutation saved by refval12

Edits are applied before B's prompt processing and used in both the plan and choice passes. Tokens come from the intervention row and are fed to both rows; no saved reference response. Each branch's choice pass uses its own plan.

Per-case checks: edits touch only result values; saved permutation reused; no A result line in B's prompts; reference row matches an ordinary fk_fv forward on the same history; intervention row matches a full-sequence replay forward on the same history; replacement on every layer of every forward; input caches unchanged; intact branches reproduce refgen12 exactly. Stops at the first failing case.

**Cell 4** runs the first case; **Cell 5** the remaining eight (resumes).

Upload `src/`, `configs/`, `scripts/`, `data/plan4_timing12.jsonl`, and the folders `results_stage_plan4_refval12/` and `results_stage_plan4_refgen12/`.

Download `/kaggle/working/results_stage_plan4_refgenval12`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_refgenval12.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_refgenval12.py").is_file() and (cand / "data" / "plan4_timing12.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_refgenval12.py and data/plan4_timing12.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input") or not (dest / "src" / "stage_plan4_refgenval12.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
for folder, names in (("results_stage_plan4_refval12", ("a_artifacts_12.jsonl", "recon12_check_summary.json", "case_checks.jsonl", "summary.json")),
                      ("results_stage_plan4_refgen12", ("evaluations.jsonl",))):
    for name in names:
        print(folder, name, (REPO / folder / name).is_file())

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_refgenval12. First case, then the rest.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_refgenval12", "--split", "plan4_timing12", "--limit", "1"])
print("refgenval12 first case exit", rc)
path = WORK / "results_stage_plan4_refgenval12" / "case_checks.jsonl"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_refgenval12" / "run" / "case_checks.jsonl"
rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
for row in rows:
    print(row["example_id"], "ok", row["ok"], "n_result", row["n_result"])
    for k, v in row["checks"].items():
        if isinstance(v, dict) and "mean_kl" in v:
            print("  ", k, {kk: v.get(kk) for kk in ("ok", "mean_kl", "argmax_agree", "labels_equal", "n_steps")})
        else:
            print("  ", k, v)
    print("  comparisons", json.dumps(row["comparisons"], indent=1))
if not all(r["ok"] for r in rows):
    raise RuntimeError("Check failed. Do not run the remaining cases.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_refgenval12", "--split", "plan4_timing12"])
print("refgenval12 exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_refgenval12"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_refgenval12" / "run"
data = json.loads((src / "summary.json").read_text(encoding="utf-8"))
print("cases", data["n_cases"], "of", data["n_expected_cases"], "| instrumentation_ok", data["instrumentation_ok"])
print("intact branches reproduce refgen12", data["intact_reproduce_refgen12"])
keys = ("labels_correct", "n_labels", "cases_all_twelve_correct", "ineligible_marked_eligible",
        "eligible_marked_ineligible", "parse_ok", "plan_truncated", "answer_truncated", "plan_lines_missing")
for b, block in data["labels"].items():
    print(b, {k: block.get(k) for k in keys})
print("\nfinal choice (separate)")
for b, block in data["final_choice"].items():
    print(b, block)
print("\nmalformed output", json.dumps(data["malformed_output"], indent=1))
print("plan label confidence", json.dumps(data["plan_label_confidence"], indent=1))
print("recovery retained", data["recovery_retained"])
print("edited vs oracle text", data["edited_vs_oracle_text"])
for name, block in data["paired_by_case"].items():
    c = block["labels_correct"]
    print(name, {k: c.get(k) for k in ("total_delta", "ci95_per_case", "cases_increased", "cases_decreased")})
for b, div in data["divergence_from_reference_row"].items():
    d = {k: v for k, v in div.items() if k != "per_case"}
    print("\ndivergence", b, d)